## Trinity Challenge Stage 2 Submission

Dataset used:
emission-spot-primary-market-auction-report-2025-data.xlsx
<br>
Downloaded directly from the EEX EUA website [Here](https://www.eex.com/en/market-data/market-data-hub/environmentals/eex-eua-primary-auction-spot-download)


In [23]:
# Setup
! pip install dimod dwave-neal numpy pandas openpyxl matplotlib pulp


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


- `dimod` + `dwave-neal`: QUBO construction and SA solver
- `numpy` + `pandas`: data handling and the EEX xlsx
- `openpyxl`: reading the `.xlsx` file with pandas
- `matplotlib`: plots for the notebook (price distribution, cover ratio, energy convergence)
- `pulp`: IP baseline solver (the classical comparison the judges require)

In [24]:
# Quick sanity check to confirm all the imports
import dimod
import neal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp

print("dimod:", dimod.__version__)
print("neal:", neal.__version__)
print("pulp:", pulp.__version__)
print(pulp.listSolvers(onlyAvailable=True))
# Should show ['PULP_CBC_CMD'] at minimum

# Quick sanity check: tiny 2-variable QUBO
bqm = dimod.BinaryQuadraticModel(vartype='BINARY')
bqm.add_variable('x0', -1.0)  # wants to be 1
bqm.add_variable('x1', -1.0)  # wants to be 1
bqm.add_interaction('x0', 'x1', 3.0)  # but penalises both being 1
# E=−x0​−x1​+3x0​x1​ is the model

sampler = neal.SimulatedAnnealingSampler()
result = sampler.sample(bqm, num_reads=100)
print("\nBest sample:", result.first.sample)
print("Energy:", result.first.energy)
# Should give x0=1, x1=0 (or x1=1, x0=0) with energy -1.0

dimod: 0.12.21
neal: 0.6.0
pulp: 3.3.2
['PULP_CBC_CMD']

Best sample: {'x0': np.int8(0), 'x1': np.int8(1)}
Energy: -1.0


In [25]:

# Load
df_raw = pd.read_excel('emission-spot-primary-market-auction-report-2025-data.xlsx', header=5)
df_raw = df_raw.dropna(subset=['Date'])

# Keep only useful columns
keep = {
    'Date': 'date',
    'Auction Name': 'auction_type',
    'Auction Price €/tCO2': 'clearing_price',
    'Minimum Bid €/tCO2': 'min_bid',
    'Maximum Bid €/tCO2': 'max_bid',
    'Mean €/tCO2': 'mean_bid',
    'Median €/tCO2': 'median_bid',
    'Auction Volume tCO2': 'supply',
    'Number of bids submitted': 'n_bids',
    'Number of successful bids': 'n_successful',
    'Average bid size': 'avg_bid_size',
    'Standard deviation of bid volume per bidder': 'std_bid_volume',
    'Cover Ratio': 'cover_ratio',
    'Total Number of Bidders': 'n_bidders',
    'Number of Successful Bidders': 'n_successful_bidders',
    'Innovation Fund\n(IF)': 'fund_IF',
    'InnoFund RRF\n(IX)': 'fund_IX',
    'Modernisation Fund\n(MF)': 'fund_MF',
    'MS RRF\n(MX)': 'fund_MX',
    'Social Climate Fund\n(SF)': 'fund_SF',
}

df = df_raw[list(keep.keys())].rename(columns=keep).copy()
df['date'] = pd.to_datetime(df['date'])

# Clean auction type labels
type_map = {
    'Auction 4. Period CAP3 EU': 'EUA',
    'Auction 4. Period DE': 'EUA_DE',
    'Auction 4. Period CAP3 PL': 'EUA_PL',
    'Auction 4. Period CAP3 NIR': 'EUAA',
}
df['auction_type'] = df['auction_type'].map(type_map).fillna(df['auction_type'])

# Summary per type
print(df.groupby('auction_type')[['clearing_price','supply','n_bids','cover_ratio','avg_bid_size','std_bid_volume']].mean().round(2))
print(f"\nTotal sessions: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

print(df_raw['Auction Name'].value_counts())
print(f"\nTotal: {df_raw['Auction Name'].nunique()} unique types")

print(df)

/home/username/Desktop/Python/Quantum_Dice/lib64/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


              clearing_price      supply  n_bids  cover_ratio  avg_bid_size  \
auction_type                                                                  
EUA                    73.36  3252834.51   98.77         1.54      51159.30   
EUAA                   77.39   796500.00   49.00         1.65      26776.00   
EUA_DE                 73.78  1633411.11   86.64         2.06      39002.20   
EUA_PL                 73.25  2101300.00   94.08         1.79      40223.88   

              std_bid_volume  
auction_type                  
EUA                258434.39  
EUAA                70230.00  
EUA_DE             171873.29  
EUA_PL             184609.36  

Total sessions: 213
Date range: 2025-01-07 to 2025-12-15
Auction Name
Auction 4. Period CAP3 EU     142
Auction 4. Period DE           45
Auction 4. Period CAP3 PL      25
Auction 4. Period CAP3 NIR      1
Name: count, dtype: int64

Total: 4 unique types
          date auction_type  clearing_price  min_bid  max_bid  mean_bid  \
0   2025

## Data Limitations
while the data for all the conducted auctions is publicily available the exact bid made by each party in the bidding is not public information we solve this by building our own bid constructor which generates bids based on the meadian and mean data which is available

In [26]:
def reconstruct_bids_multisession(sessions, package_fraction=0.3, seed=42):
    """
    Reconstruct bids across all 4 auction types from session statistics.
    
    Package bids (Option 2):
    EUAA is aviation-sector only. Any bidder in EUAA also likely
    participates in EUA for their non-aviation installations.
    We model this by linking a fraction of EUAA bids to EUA bids
    as complementary package pairs — accepting both gives bonus w_ik.
    
    package_fraction: fraction of EUAA bidders assumed to also bid EUA
    """
    rng = np.random.default_rng(seed)
    all_bids = []

    for tau, session in sessions.items():
        n       = int(session['n_bids'])
        mu_q    = session['avg_bid_size']
        sigma_q = session['std_bid_volume']
        p_clear = session['clearing_price']
        p_min   = session['min_bid']
        p_max   = session['max_bid']
        supply  = session['supply']

        # Quantities: log-normal from real mean and std
        var   = sigma_q**2
        mu_ln = np.log(mu_q**2 / np.sqrt(var + mu_q**2))
        sg_ln = np.sqrt(np.log(1 + var / mu_q**2))
        quantities = rng.lognormal(mu_ln, sg_ln, n)

        # Prices: triangular between real min/max, peak at clearing price
        prices = rng.triangular(p_min, p_clear, p_max, n)

        bids = pd.DataFrame({
            'quantity': quantities,
            'price':    prices,
            'value':    quantities * prices,
            'type':     tau,
            'supply':   supply,
            'package_partner': -1,   # -1 means no package link
            'package_bonus':    0.0,
        })

        # Force cover ratio to match real session
        real_cover    = session['cover_ratio']
        current_cover = bids['quantity'].sum() / supply
        bids['quantity'] *= (real_cover / current_cover)
        bids['value']     = bids['quantity'] * bids['price']

        all_bids.append(bids)

    # Concatenate all sessions into one bid pool
    combined = pd.concat(all_bids, ignore_index=True)

    # --- Package bid linking ---
    # EUAA bids represent aviation operators.
    # A fraction of them also hold EUA bids for non-aviation installations.
    # These pairs have complementary value: w_ik > 0 when both accepted.
    #
    # w_ik is modelled as a fraction of the smaller bid's value —
    # the "synergy premium" for covering full compliance in one allocation.
    euaa_idx = combined[combined['type'] == 'EUAA'].index.tolist()
    eua_idx  = combined[combined['type'] == 'EUA'].index.tolist()

    n_packages = int(len(euaa_idx) * package_fraction)
    # Pick random EUAA bids as package participants
    euaa_partners = rng.choice(euaa_idx, size=n_packages, replace=False)
    # Match each to a random EUA bid
    eua_partners  = rng.choice(eua_idx,  size=n_packages, replace=False)

    for euaa_i, eua_k in zip(euaa_partners, eua_partners):
        w = 0.15 * min(combined.loc[euaa_i, 'value'],
                       combined.loc[eua_k,  'value'])
        combined.loc[euaa_i, 'package_partner'] = eua_k
        combined.loc[eua_k,  'package_partner'] = euaa_i
        combined.loc[euaa_i, 'package_bonus']   = w
        combined.loc[eua_k,  'package_bonus']   = w

    return combined

sessions = {
    tau: df[df['auction_type'] == tau].iloc[len(df[df['auction_type'] == tau])//2]
    for tau in ['EUA', 'EUA_DE', 'EUA_PL', 'EUAA']
}

# Build multi-session bid pool
all_bids = reconstruct_bids_multisession(sessions)

print(f"Total bids: {len(all_bids)}")
print(all_bids.groupby('type')[['quantity','value','supply']].agg({
    'quantity': ['mean','sum'],
    'value':    'mean',
    'supply':   'first'
}).round(0))
print(f"\nPackage pairs: {(all_bids['package_partner'] != -1).sum() // 2}")
print(f"Avg package bonus: €{all_bids[all_bids['package_bonus']>0]['package_bonus'].mean():,.0f}")

Total bids: 321
       quantity                 value   supply
           mean        sum       mean    first
type                                          
EUA     45112.0  4511245.0  3263908.0  3245500
EUAA    26821.0  1314225.0  1930094.0   796500
EUA_DE  38925.0  3503260.0  3438688.0  1607000
EUA_PL  43472.0  3564700.0  3792499.0  2072500

Package pairs: 14
Avg package bonus: €117,031


In [27]:
def build_qubo(bids, A=1.0, C_mult=3.0):
    """
    QUBO matrix Q for welfare-maximising auction clearing with package bids.

    Full Hamiltonian:
        H = H_obj + H_pkg + H_cap

    H_obj = -A * sum_i v_i x_i
        Reward for accepting each bid. Diagonal terms only.

    H_pkg = -A * sum_{(i,k) in packages} w_ik * x_i * x_k
        Bonus for accepting complementary pairs together.
        w_ik > 0 so this term is negative (rewarding) when both x_i=x_k=1.
        This adds off-diagonal attractive coupling J_ik = -A*w_ik.
        This is what makes the problem beyond knapsack.

    H_cap = C * sum_tau ( sum_{i in tau} q_i x_i - S_tau )^2
        Quadratic penalty for exceeding supply per type.
        Expanding the square gives:
          diagonal:     C*(q_i^2 - 2*S*q_i) per bid
          off-diagonal: 2*C*q_i*q_k per same-type pair
        C must dominate A*v_max to ensure feasibility at ground state.

    All values and quantities normalised to [0,1] before building Q
    so that penalty coefficients are O(1) not O(1e18).
    """
    bids  = bids.reset_index(drop=True)
    n     = len(bids)

    # --- Normalise per type so each type's scale is independent ---
    v_raw = bids['value'].values.copy()
    q_raw = bids['quantity'].values.copy()
    v_scale = v_raw.max()
    q_scale = q_raw.max()

    v = v_raw / v_scale
    q = q_raw / q_scale

    v_max  = v.max()                          # max normalized bid value
    w_max  = bids['package_bonus'].max() / v_scale  # max normalized bonus
    C      = C_mult * (v_max + w_max + 1.0)  # dominates both objective terms

    Q = np.zeros((n, n))

    # H_obj: diagonal reward
    for i in range(n):
        Q[i, i] -= A * v[i]

    # H_pkg: off-diagonal attractive coupling for package pairs
    # J_ik = -A * w_ik (negative = attractive = rewards joint acceptance)
    for i in range(n):
        partner = bids.loc[i, 'package_partner']
        if partner != -1 and partner > i:
            w_ik = bids.loc[i, 'package_bonus'] / v_scale  # normalise bonus
            Q[i, int(partner)] -= A * w_ik

    # H_cap: quadratic supply constraint per auction type
    for tau in bids['type'].unique():
        idx = bids[bids['type'] == tau].index.tolist()
        S   = bids.loc[idx[0], 'supply'] / q_scale   # normalised supply

        for i in idx:
            Q[i, i] += C * q[i]**2 - 2 * C * S * q[i]
            for k in idx:
                if k > i:
                    Q[i, k] += 2 * C * q[i] * q[k]

    return Q, {'A': A, 'C': C, 'C_mult': C_mult,
                'v_scale': v_scale, 'q_scale': q_scale,
                'n_packages': (bids['package_partner'] != -1).sum() // 2}


Q, params = build_qubo(all_bids)
print("QUBO shape:", Q.shape)
print("Params:", {k: f"{v:.4g}" for k, v in params.items()})
print(f"Q diagonal range:  [{Q.diagonal().min():.4g}, {Q.diagonal().max():.4g}]")
print(f"Q off-diag nonzeros: {np.count_nonzero(Q - np.diag(Q.diagonal()))}")

QUBO shape: (321, 321)
Params: {'A': '1', 'C': '6.009', 'C_mult': '3', 'v_scale': '9.298e+07', 'q_scale': '1.048e+06', 'n_packages': '14'}
Q diagonal range:  [-28.94, -0.002869]
Q off-diag nonzeros: 13466


In [29]:
def solve_sa(Q, bids, num_reads=200, num_sweeps=1000):
    bids = bids.reset_index(drop=True)
    n    = len(bids)

    bqm = dimod.BinaryQuadraticModel(vartype='BINARY')
    for i in range(n):
        bqm.add_variable(i, Q[i, i])
    for i in range(n):
        for k in range(i+1, n):
            if Q[i, k] != 0.0:
                bqm.add_interaction(i, k, Q[i, k])

    sampler = neal.SimulatedAnnealingSampler()
    result  = sampler.sample(bqm, num_reads=num_reads, num_sweeps=num_sweeps)

    best = result.first.sample
    x    = np.array([best[i] for i in range(n)])

    accepted     = bids[x == 1].copy()
    welfare      = accepted['value'].sum()
    pkg_bonus    = sum(
        bids.loc[i, 'package_bonus']
        for i in accepted.index
        if bids.loc[i, 'package_partner'] in accepted.index
        and bids.loc[i, 'package_partner'] != -1
    )

    # Feasibility per type
    feasible = all(
    accepted[accepted['type'] == tau]['quantity'].sum()
    <= bids[bids['type'] == tau]['supply'].iloc[0]
    for tau in bids['type'].unique()
    )

    return {
        'x': x, 'welfare': welfare, 'pkg_bonus': pkg_bonus,
        'volume_used': accepted['quantity'].sum(),
        'n_accepted': int(x.sum()), 'feasible': feasible,
        'energy': result.first.energy,
    }


def solve_ip(bids):
    bids  = bids.reset_index(drop=True)
    n     = len(bids)
    prob  = pulp.LpProblem("auction_clearing", pulp.LpMaximize)
    x     = [pulp.LpVariable(f"x_{i}", cat='Binary') for i in range(n)]

    # Base welfare
    base = pulp.lpSum(bids.loc[i, 'value'] * x[i] for i in range(n))

    # Package bonus: extra value when both i and partner k are accepted
    # z_ik = x_i * x_k linearised via auxiliary variable z_ik <= x_i, z_ik <= x_k
    pkg_vars = {}
    pkg_bonus_terms = []
    seen = set()
    for i in range(n):
        k = int(bids.loc[i, 'package_partner'])
        if k != -1 and (i, k) not in seen and (k, i) not in seen:
            seen.add((i, k))
            z = pulp.LpVariable(f"z_{i}_{k}", cat='Binary')
            pkg_vars[(i, k)] = z
            prob += z <= x[i]
            prob += z <= x[k]
            prob += z >= x[i] + x[k] - 1
            pkg_bonus_terms.append(bids.loc[i, 'package_bonus'] * z)

    prob += base + pulp.lpSum(pkg_bonus_terms)

    # Supply constraint per type
    for tau in bids['type'].unique():
        idx = bids[bids['type'] == tau].index.tolist()
        S   = bids.loc[idx[0], 'supply']
        prob += pulp.lpSum(bids.loc[i, 'quantity'] * x[i] for i in idx) <= S

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    x_val    = np.array([pulp.value(x[i]) for i in range(n)])
    accepted = bids[x_val == 1].copy()
    pkg_bonus = sum(pulp.value(z) * bids.loc[i, 'package_bonus']
                    for (i, k), z in pkg_vars.items())

    return {
        'welfare':     pulp.value(prob.objective),
        'pkg_bonus':   pkg_bonus,
        'volume_used': accepted['quantity'].sum(),
        'n_accepted':  int(x_val.sum()),
        'feasible': all(
    accepted[accepted['type'] == tau]['quantity'].sum()
    <= bids[bids['type'] == tau]['supply'].iloc[0]
    for tau in bids['type'].unique()),
    }


# --- Run both solvers ---
sa_result = solve_sa(Q, all_bids)
ip_result = solve_ip(all_bids)

print("=== SA ===")
print(f"Feasible:    {sa_result['feasible']}")
print(f"Accepted:    {sa_result['n_accepted']} / {len(all_bids)}")
print(f"Welfare:     €{sa_result['welfare']:,.0f}")
print(f"Pkg bonus:   €{sa_result['pkg_bonus']:,.0f}")
print(f"Energy:      {sa_result['energy']:.4g}")

print("\n=== IP ===")
# print(f"Status:      {ip_result['status']}")
print(f"Feasible:    {ip_result['feasible']}")
print(f"Accepted:    {ip_result['n_accepted']} / {len(all_bids)}")
print(f"Welfare:     €{ip_result['welfare']:,.0f}")
print(f"Pkg bonus:   €{ip_result['pkg_bonus']:,.0f}")

print("\n=== Gap ===")
gap = 100 * (1 - sa_result['welfare'] / ip_result['welfare'])
print(f"Welfare gap: {gap:.2f}%")
print(f"SA captures {100-gap:.1f}% of optimal welfare")

accepted_sa = all_bids[sa_result['x'].astype(bool)].copy()
for tau in all_bids['type'].unique():
    sa_vol = accepted_sa[accepted_sa['type'] == tau]['quantity'].sum()
    supply = all_bids[all_bids['type'] == tau]['supply'].iloc[0]
    print(f"{tau}: volume={sa_vol:,.0f}  supply={supply:,.0f}  ratio={sa_vol/supply:.2f}")

=== SA ===
Feasible:    False
Accepted:    159 / 321
Welfare:     €673,180,010
Pkg bonus:   €2,745,327
Energy:      -105.8

=== IP ===
Feasible:    True
Accepted:    161 / 321
Welfare:     €650,559,073
Pkg bonus:   €1,462,059

=== Gap ===
Welfare gap: -3.48%
SA captures 103.5% of optimal welfare
EUA: volume=3,314,996  supply=3,245,500  ratio=1.02
EUA_DE: volume=1,699,448  supply=1,607,000  ratio=1.06
EUA_PL: volume=2,158,871  supply=2,072,500  ratio=1.04
EUAA: volume=868,954  supply=796,500  ratio=1.09
